In [77]:
import pandas as pd
import numpy as np

In [84]:
        
data = pd.read_excel(r"/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/2022-12-SGFTFN_MICA_unprocessed.xlsx",header=0,index_col=0)
# data = pd.concat([data,data1],axis=0)
# data = pd.read_excel(r"/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/2022-12-SGFTFN_CLAY_MINERALS_unprocessed.xlsx",header=0,index_col=0)

oxide = pd.read_excel(r"/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/processed minerals/oxide_data.xlsx", sheet_name="Sheet1",index_col=0,header=0)
oxlist = ["SiO2", "TiO2", "Al2O3", "Cr2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O","P2O5"]

def wt_to_mol(data, oxide = oxide):
    data[data<=2] = 0
    oxlist1 = oxlist.copy()
    oxide = oxide.T[oxlist1].iloc[0, :].to_numpy()
    data = normalize(data)
    [r, c] = data.shape
    data_f = np.empty((r, c))
    for i in range(0, c):
        data_f[:, i] = data[:, i] / oxide[i]
    data
    data_f = normalize(data_f)
    return(data_f.round(2))


def normalize(data):
    [r, c] = data.shape
    a = data.sum(axis=1).reshape((len(data), 1))
    data_formatted = ((data*100)/a).round(1)
    return(data_formatted)


def cat_calc(data1,oxide_list):
    total = data1.columns.get_loc("Total")
    o_no = data1.loc[:,"Oxygen_no"]
    data = data1.iloc[:,:total].copy()
    oxide = oxide_list.loc[data.columns]
    data = data.div(oxide['Mol. Wt.'].values,axis=1).round(3)
    data = data.mul(oxide['O_no'].values,axis=1).round(3)
    norm = o_no.div(data.sum(axis=1)).round(3)
    data = data.mul(norm.values,axis=0).round(3)
    data = data.mul(oxide['Cat_per_o'].values,axis=1).round(3)
    total = data.sum(axis=1).round(3)
    data.columns = oxide['Cation'].values
    data['Cation_Total'] = total
    return data.round(3)


In [85]:
data2 = data[['SIO2(WT%)','TIO2(WT%)','AL2O3(WT%)','CR2O3(WT%)','FE2O3T(WT%)', 'FE2O3(WT%)', 'FEOT(WT%)','FEO(WT%)', 'MNO(WT%)','MGO(WT%)','CAO(WT%)', 'NA2O(WT%)','K2O(WT%)','P2O5(WT%)','MINERAL']]

In [86]:
data2.MINERAL.unique()

array(['PHLOGOPITE', 'BIOTITE', 'MICA', 'SERICITE', 'CELADONITE',
       'MUSCOVITE', 'MARGARITE', 'GLAUCONITE', 'ANNITE', 'PHENGITE',
       'PHENGITE-MUSCOVITE', 'ZINNWALDITE', 'PARAGONITE', 'LEPIDOLITE',
       'SIDEROPHYLLITE', 'YANGZHUMINGITE', 'HYDROMUSCOVITE', 'HYDROMICA',
       nan], dtype=object)

In [88]:
data_cleaned = data2.loc[~data2['SIO2(WT%)'].isna(),:]
# data_px = data_cleaned[(data_cleaned["MINERAL"]=="MAGNETITE") | (data_cleaned["MINERAL"]=="TITANO-MAGNETITE")]
data_px = data_cleaned.copy()
data_px = data_px.iloc[:,:-1].apply(pd.to_numeric,args=('coerce',)).astype('float')

cond = (data_px['FEOT(WT%)'].isna()) & (~data_px['FE2O3T(WT%)'].isna())
data_px.loc[cond,'FEOT(WT%)'] = data_px.loc[cond,'FE2O3T(WT%)']*.8998
cond = (data_px['FEOT(WT%)'].isna()) & (data_px['FE2O3T(WT%)'].isna())
data_px.loc[cond,'FEOT(WT%)'] = data_px.loc[cond,'FEO(WT%)'] + (data_px.loc[cond,'FE2O3(WT%)']*.8998)
data_px.loc[~data_px['FEOT(WT%)'].isna(),:]
data_px.pop("FE2O3(WT%)")
data_px.pop("FE2O3T(WT%)")
data_px.pop("FEO(WT%)")
data_px['Mineral'] = data_cleaned['MINERAL']
data_px.columns = ["SiO2", "TiO2", "Al2O3", "Cr2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O","P2O5",'Mineral']
mineral = data_px['Mineral']
data_px = data_px.iloc[:,:-1].apply(pd.to_numeric,args=('coerce',)).astype('float')
data_px2 = data_px.fillna(0)
total = data_px2.sum(axis=1)
data_px2['Mineral'] = mineral
data_px2 = data_px2[(total>=80) & (total<=101)]
mineral = data_px2.pop("Mineral")
col = data_px2.columns
ind = data_px2.index
data_px2 = pd.DataFrame(wt_to_mol(data_px2.to_numpy(),oxide),columns = col,index=ind)
# data_px2 = data

non_essential_sum = data_px2[["TiO2","CaO", "Na2O", "K2O","P2O5"]].sum(axis=1)
data_px2 = data_px2[non_essential_sum<=9]
m = data_px2[["FeO","MnO","MgO"]].sum(axis=1)
data_px2 = data_px2[(data_px2.SiO2 <= 34) & (m >=54) ]

# data_px2 = data_px2[ (m >= 30) & (m <= 100)]
# data_px2 = data_px2[(data_px2.Al2O3 + data_px2.Cr2O3 >= 49) & (data_px2.Al2O3 + data_px2.Cr2O3 <= 51)]
# data_px2['P2O5'] = 0
data_px2['Mineral'] = "Chl"
data_px2['Al2O3'] = data_px2['Al2O3'] + data_px2['Cr2O3']
data_px2.pop("Cr2O3")
# data_px2.to_excel("/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/processed minerals/molar tables/new data/Chl_processed_mol.xlsx")
data_px2['M'] = data_px2[["FeO","MnO","MgO"]].sum(axis=1)
data_px2.sort_values("M",ascending=False)


,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,Mineral,M
CITATION,,,,,,,,,,,,
[20064] ROSENBAUM J. M. (1993),24.7,0.0,0.0,8.5,3.3,61.0,0.0,0.0,2.6,0.0,Chl,72.8
[15288] DONNELLY C. L. (2011),20.6,0.0,6.1,6.2,0.0,59.3,5.1,0.0,2.7,0.0,Chl,65.5
[22616] FULOP A. (2018),32.4,0.0,6.8,12.1,0.0,48.7,0.0,0.0,0.0,0.0,Chl,60.8
[22616] FULOP A. (2018),33.5,0.0,5.9,12.1,0.0,48.5,0.0,0.0,0.0,0.0,Chl,60.6
[15288] DONNELLY C. L. (2011),27.3,2.2,8.7,6.7,0.0,49.6,0.0,0.0,5.4,0.0,Chl,56.3
[22403] HAO LU-LU (2018),29.7,4.3,6.2,35.1,0.0,20.1,0.0,0.0,4.6,0.0,Chl,55.2
[23994] AZADBAKHT Z. (2020),33.7,0.0,11.1,23.0,0.0,32.1,0.0,0.0,0.0,0.0,Chl,55.1
[23994] AZADBAKHT Z. (2020),31.5,0.0,13.5,47.5,0.0,7.5,0.0,0.0,0.0,0.0,Chl,55.0
[23994] AZADBAKHT Z. (2020),31.7,0.0,13.5,45.4,0.0,9.4,0.0,0.0,0.0,0.0,Chl,54.8


In [60]:
cond = (data_cleaned['FEOT(WT%)'].isna()) & (~data_cleaned['FE2O3T(WT%)'].isna())
# cond = (data_cleaned['FEOT(WT%)'].isna()) & (data_cleaned['FE2O3T(WT%)'].isna())

data_cleaned.loc[cond,'FE2O3T(WT%)']

CITATION
[742] CELESTINO SILVA L. (1987)     0.30
[742] CELESTINO SILVA L. (1987)     2.29
[742] CELESTINO SILVA L. (1987)     2.03
[3570] NIIDA K. (1975)              3.54
[3570] NIIDA K. (1975)              3.46
                                   ...  
[24618] MOHAMMADI N. (2021)        18.91
[24618] MOHAMMADI N. (2021)        14.84
[24618] MOHAMMADI N. (2021)        15.91
[24618] MOHAMMADI N. (2021)        13.82
[24618] MOHAMMADI N. (2021)         9.81
Name: FE2O3T(WT%), Length: 210, dtype: float64